In [1]:
import os
import hashlib
from dotenv import load_dotenv
 
from pinecone import Pinecone, ServerlessSpec
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_pinecone import PineconeVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

/var/folders/d_/47lp0l293sx5khzrnt9d5_2m0000gn/T/ipykernel_68771/3775987701.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
load_dotenv()

True

In [3]:
PDF_PATH = "AI Agenda_Updated'26.pdf"   
NAMESPACE = "pdf-docs"   

In [4]:
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(f"Loaded {len(pages)} pages from {PDF_PATH}")

Loaded 35 pages from AI Agenda_Updated'26.pdf


In [ ]:
pages

[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2025-12-10T06:42:25+00:00', 'title': "AI Agenda_Updated'26", 'moddate': '2025-12-10T06:42:17+00:00', 'keywords': 'DAF026VVRhc,BAC39NGjlTo,0', 'author': 'Enquiry BEPEC', 'source': "AI Agenda_Updated'26.pdf", 'total_pages': 35, 'page': 0, 'page_label': '1'}, page_content='BECOME AN AI, AGENTIC AI EXPERT IN 7MONTHS\n"Our 100% Job Guaranteed Placement Formula for\nyour Career Transition" \nB E P E C  S O L U T I O N S  P V T . L T D\nw w w . b e p e c . i n\nW h a t s a p p  + 9 1 9 6 4 4 4 6 6 2 2 2\nFULL STACK ARTIFICIAL INTELLIGENCE \nCOURSE WITH GEN AI'),
 Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2025-12-10T06:42:25+00:00', 'title': "AI Agenda_Updated'26", 'moddate': '2025-12-10T06:42:17+00:00', 'keywords': 'DAF026VVRhc,BAC39NGjlTo,0', 'author': 'Enquiry BEPEC', 'source': "AI Agenda_Updated'26.pdf", 'total_pages': 35, 'page': 1, 'page_label': '2'}, page_content='ABOUT BEPEC

In [6]:
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "langchain-demo"

In [ ]:
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1536,          
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
 
index = pc.Index(index_name)

In [8]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")  # 1536 dims
vector_store = PineconeVectorStore(index=index, embedding=embeddings, namespace=NAMESPACE)

In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks")

Split into 50 chunks


In [10]:
ids = [
    hashlib.md5(f"{c.metadata['source']}::{c.page_content}".encode()).hexdigest()
    for c in chunks
]

In [11]:
vector_store.add_documents(chunks, ids=ids, batch_size=100)
print("Stored in Pinecone")

Stored in Pinecone


In [12]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [13]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [14]:
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. Answer the question using ONLY the context below. "
     "If the answer is not in the context, say you don't know. "
     "Mention the page number(s) you used.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])

In [15]:
def format_docs(docs):
    # Include the page number so the LLM can cite it (PyPDFLoader pages start at 0)
    return "\n\n".join(
        f"[Page {d.metadata.get('page', 0) + 1}]\n{d.page_content}" for d in docs
    )
 

In [16]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [17]:
rag_chain.invoke("What is covered in Python")

'The Python agenda covers the following topics:\n\n- Introduction to Python\n- Why Python, Value, Variable, Function, Library\n- IDE in Python, Different Data Types\n- List, Tuple, Set & Dictionary Overview\n- Different List Methods\n- Different Tuple Methods\n- Set & Frozenset\n- Dictionary & String Manipulations\n- Overview on Loops, If Statements, UDFs, Escape Sequences, Lambda\n- Types of Operators, Conditional Statements\n- While Loop, List Comprehension, Break, Continue, Arguments\n- Functions, Escape Sequences, Lambda Functions\n- Hackathon-1\n- Introduction to OOPS\n- Instance Variable, Class Variable, Class Method\n- Association vs Composition & Aggregation\n- OOP Concepts\n- Encapsulation, Inheritance\n- Polymorphism, Method Overloading, Method Overriding\n- Introduction to Pandas\n- Data Analysis using Pandas\n- Introduction to Numpy\n- Different Numpy Commands\n- Introduction to Data Visualisation\n- Data Visualisation using Matplotlib\n- Data Visualisation using Seaborn\n-